<h2>MonReader</h2>

We collected page flipping video from smart phones and labelled them as flipping and not flipping.

We clipped the videos as short videos and labelled them as flipping or not flipping. The extracted frames are then saved to disk in a sequential order with the following naming structure: VideoID_FrameNumber

#### **Goal(s):**

Predict if the page is being flipped using a single image.

#### **Success Metrics:**

Evaluate model performance based on F1 score, the higher the better.

#### **Current Challenges:**

Predict if a given sequence of images contains an action of flipping.

In [1]:
import numpy as np
import pandas as pd

<h3>EDA</h3>

In [3]:
from pathlib import Path

Each image in the flip and notflip folders has the following naming/numbering convention: 00wx_0000000yz.jpg, where wx is the clip number, and yz is the frame number. 

Let's count the number of clips, and the number of frames in each clip:

In [5]:
def count_frames_per_clip(data_dir) -> pd.DataFrame:
    """
    Count frames per clip in the dataset.

    Expects: data_dir / <split> / <label> / <VideoID>_<FrameNumber>.jpg
    e.g.     data/raw / training  / flip   / 0001_000000020.jpg

    VideoID numbering resets inside each (split, label) folder, so a clip is
    only uniquely identified by the (split, label, video_id) triple, not by
    video_id alone.

    Returns one row per clip: split, label, video_id, frame_count,
    min_frame, max_frame (raw frame numbers aren't contiguous within a clip,
    so max_frame - min_frame + 1 can be larger than frame_count).
    """
    data_dir = Path(data_dir)
    rows = []
    for split_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        for label_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            counts, frame_min, frame_max = {}, {}, {}
            for img_path in label_dir.glob("*.jpg"):
                video_id, _, frame_str = img_path.stem.partition("_")
                frame_number = int(frame_str)
                counts[video_id] = counts.get(video_id, 0) + 1
                frame_min[video_id] = min(frame_min.get(video_id, frame_number), frame_number)
                frame_max[video_id] = max(frame_max.get(video_id, frame_number), frame_number)
            for video_id, frame_count in counts.items():
                rows.append({
                    "split": split_dir.name,
                    "label": label_dir.name,
                    "video_id": video_id,
                    "frame_count": frame_count,
                    "min_frame": frame_min[video_id],
                    "max_frame": frame_max[video_id],
                })
    return (
        pd.DataFrame(rows)
        .sort_values(["split", "label", "video_id"])
        .reset_index(drop=True)
    )


# usage
df = count_frames_per_clip("data/raw/training/flip")
print(df.groupby(["split", "label"])["frame_count"].agg(["count", "mean", "min", "max"]))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data\\raw\\training\\flip'